# LC 994 — Rotting Oranges
**Difficulty:** Medium | **Pattern:** Multi-Source BFS

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> All rotten oranges spread rot
<em>simultaneously</em>. Seed the BFS queue with every rotten
orange at once (time=0), then let BFS count the minutes. Each
level of BFS is one minute. If any fresh orange remains after
BFS, return -1.
</div>

## Official Problem Statement

You are given an `m x n` integer grid where:
- `0` = empty cell
- `1` = fresh orange
- `2` = rotten orange

Every minute, any fresh orange **4-directionally adjacent** to a
rotten orange becomes rotten.

Return the **minimum number of minutes** until no fresh orange
remains. If this is impossible, return `-1`.

**Constraints:**
- `m == grid.length`, `n == grid[i].length`
- `1 <= m, n <= 10`
- `grid[i][j]` is `0`, `1`, or `2`

## What This Is Actually Asking

This is a simulation problem — but simulating one rotten orange
at a time would be wrong because all rotten ones spread at once.
Multi-source BFS handles simultaneous spread naturally: add all
starting sources to the queue together, and each BFS "level"
represents one time step. Count fresh oranges before BFS; if
the count reaches zero during BFS the answer is the elapsed
minutes, otherwise it is -1.

## Walk Through an Example by Hand

```
Initial grid:
  2 1 1
  1 1 0
  0 1 1

fresh=6, queue=[(0,0)]

Minute 1: spread from (0,0)
  → rot (0,1) and (1,0); fresh=4
  grid:
  2 2 1
  2 1 0
  0 1 1

Minute 2: spread from (0,1),(1,0)
  → rot (0,2),(1,1); fresh=2
  grid:
  2 2 2
  2 2 0
  0 1 1

Minute 3: spread from (0,2),(1,1)
  → rot (2,1); fresh=1

Minute 4: spread from (2,1)
  → rot (2,2); fresh=0  ← done

Answer = 4
```

## The Picture

```
Multi-source BFS — all sources seeded at minute 0:

  queue = deque([(r,c,0) for all rotten cells])
  OR keep separate time counter:
  queue = deque([(r,c) for all rotten cells])
  time = 0

BFS levels = minutes:

  t=0: [ (0,0) ]              ← rotten at start
  t=1: [ (0,1), (1,0) ]       ← freshes adjacent to t=0
  t=2: [ (0,2), (1,1) ]       ← freshes adjacent to t=1
  t=3: [ (2,1) ]
  t=4: [ (2,2) ]  fresh=0 ✓

Directions: [(-1,0),(1,0),(0,-1),(0,1)]  (4-directional)

Key check after BFS:
  if fresh > 0: return -1
  else:         return time
```

## When To Use This Pattern

- When multiple sources **spread simultaneously** through a
  grid, think **multi-source BFS** (seed all sources at once).
- When asked for the **minimum time** for something to reach
  all cells, think **BFS level = one time unit**.
- When some cells are unreachable and the answer might be -1,
  think **count targets before BFS, check after**.
- When spread is 4-directional (not diagonal), think
  `[(-1,0),(1,0),(0,-1),(0,1)]`.
- When the grid is small (≤10×10), even DFS would pass, but
  BFS is conceptually correct.

## The Approach

Count fresh oranges. Add all rotten orange coordinates to the
BFS queue. Process the queue level by level; each level
increments the minute counter and rotates every adjacent fresh
orange, decrementing the fresh count. After BFS, if fresh is
zero return the elapsed minutes; otherwise return -1.

In [ ]:
from typing import List
from collections import deque

In [ ]:
def test_harness(func):
    cases = [
        # (grid, expected)
        (
            [[2,1,1],[1,1,0],[0,1,1]],
            4
        ),
        (
            [[2,1,1],[0,1,1],[1,0,1]],
            -1   # bottom-left isolated
        ),
        (
            [[0,2]],
            0    # no fresh oranges
        ),
        (
            [[1]],
            -1   # single fresh, no rotten
        ),
        (
            [[2]],
            0    # single rotten
        ),
        (
            [[2,2],[1,1]],
            1
        ),
    ]
    passed = 0
    for grid, expected in cases:
        import copy
        result = func(copy.deepcopy(grid))
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"  {status}: grid={grid} "
                f"=> got {result}, want {expected}"
            )
    print(f"\nSummary: {passed}/{len(cases)} passed")

In [ ]:
def oranges_rotting(grid: List[List[int]]) -> int:
    """
    Return minutes until all oranges rot, or -1 if impossible.

    Strategy: Multi-source BFS.
      1. Seed queue with all rotten cells; count fresh.
      2. BFS level by level; each level = 1 minute.
      3. Each rotten neighbour infects adjacent fresh cells.
      4. If fresh == 0 at end: return time. Else: return -1.

    Args:
        grid: m x n grid of 0 (empty), 1 (fresh), 2 (rotten)
    Returns:
        Minimum minutes to rot all oranges, or -1.
    """
    # Debug: print initial state
    # rows, cols = len(grid), len(grid[0])
    # print(f"Grid {rows}x{cols}")

    pass

    # Debug: print fresh count after BFS
    # print(f"BFS done: fresh={fresh}, time={time}")

In [ ]:
# Uncomment and run when solution is ready
# test_harness(oranges_rotting)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| BFS per rotten cell (wrong) | O(m·n·k) | O(m·n) | Incorrect order |
| Simulation (re-scan each min)| O((m·n)²) | O(1) | Correct, slow |
| **Multi-source BFS** | **O(m·n)** | **O(m·n)** | Optimal |

## Real World Connection

At **Citi**, fraud detection propagates alerts from known
fraudulent accounts to their transaction neighbours —
multi-source BFS discovers all at-risk accounts in minimum
hops. In **AWS**, health-check failures in a microservice mesh
cascade outward; multi-source BFS models the blast radius in
O(nodes+edges). For a **data engineer**, tracking how a bad
upstream data feed corrupts downstream pipeline nodes uses this
same simultaneous-spread BFS. Any "contagion from multiple
origins" problem is a multi-source BFS in disguise.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra